# Music Annotation Project V8.0c: 65×65 Context Ablation

**Runtime:** GPU. Select **Runtime → Run all**.

## Narrow scientific question

Does increasing the observable spatial context from **33×33** to **65×65** improve source-group-separated out-of-fold discrimination and the safe coverage frontier, when everything else is held constant?

## Controlled design

| Element | Control | Treatment |
|---|---|---|
| Input context | 33×33 | 65×65 |
| Channels | Same 11 observable channels | Same 11 observable channels |
| CNN | Same fixed compact architecture | Same fixed compact architecture |
| Labels | Same PRINT / ANNOTATION rows | Same rows |
| Sampling | Same centre pixels | Same centre pixels |
| Folds | Same authoritative source-group folds | Same folds |
| Training schedule | Same optimiser, epochs, loss and seed | Same |
| Calibration | Same cross-fitted isotonic method | Same |
| Decision rule | Same registered rule | Same |

The only intended causal change is **patch context size**. This notebook does not add pages, alter labels, change the loss, search architectures, lower thresholds, use independent data, or authorise promotion.

The notebook asserts the V8.0b fold-map hash where the same roster, manifest and software split behaviour reproduce it. A mismatch stops the run for audit rather than silently creating a different comparison.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib


## 1. Frozen constants

Do not modify these after inspecting V8.0c outcomes. A change requires a separately versioned experiment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc
import cv2, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0c'
V75_RUN_OVERRIDE=None

SEED=42
PATCH_CONTROL=33
PATCH_TREATMENT=65
DARK_THRESHOLD=200
TRUTH_TOLERANCE=1
MAX_PER_PAGE_CLASS=25000
BATCH=512
EPOCHS=12
LR=1e-3
WEIGHT_DECAY=1e-4
PRINT_THRESHOLD=.995
ANNOTATION_THRESHOLD=.995
PRINT_MARGIN=.990
THRESHOLD_GRID=np.unique(np.r_[np.linspace(.90,.99,10),np.linspace(.991,.999,9),.9995,.9999])
EXPECTED_V8B_FOLD_MAP_SHA256='ae1ac4557c1ffab1371094f3fe1c818fd0f652bb271512830ee06da55fc4867b'
EXPECTED_V8B_MANIFEST_SHA256='8ad9490658edcf898da7b04b2820d858a12bf4ec2ce4ab3a722d762a2831bcd7'

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Select a GPU runtime.')
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0c_65px_context_ablation'
OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; MODELS=OUT/'models'; AUDIT=OUT/'audit'; CHARTS=OUT/'charts'
for p in [OUT,TABLES,MODELS,AUDIT,CHARTS]: p.mkdir(parents=True,exist_ok=False)
print('Device:',DEVICE)
print('Output:',OUT)


## 2. Helpers and authoritative input gate


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def wilson_upper(errors,n,z=1.96):
    if n==0: return np.nan
    p=errors/n; den=1+z*z/n; centre=p+z*z/(2*n)
    radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return (centre+radius)/den

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('Complete V7.5 development run not found.')
if not MANIFEST_PATH.exists(): raise RuntimeError(f'Missing {MANIFEST_PATH}')
if sha256_file(MANIFEST_PATH)!=EXPECTED_V8B_MANIFEST_SHA256:
    raise RuntimeError('development_score_manifest.csv differs from the V8.0b manifest hash. Version and audit the change before V8.0c.')

manifest=pd.read_csv(MANIFEST_PATH,dtype=str)
required={'page','source_group','score_title','edition_or_source','acquisition_notes'}
missing=required-set(manifest.columns)
if missing: raise RuntimeError(f'Manifest missing columns: {sorted(missing)}')
if manifest.page.duplicated().any(): raise RuntimeError('Manifest contains duplicate pages.')
if manifest.source_group.isna().any() or (manifest.source_group.str.strip()=='').any():
    raise RuntimeError('Manifest source_group values must be complete.')

status=pd.read_csv(V75/'tables'/'cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean')
observed_map=image_map(ROOT/'aligned')
pages=sorted(set(cache)&set(clean_map)&set(observed_map))
if set(manifest.page)!=set(pages):
    raise RuntimeError(f'Manifest/V7.5 roster mismatch. Missing={sorted(set(pages)-set(manifest.page))[:20]} Extra={sorted(set(manifest.page)-set(pages))[:20]}')
meta=manifest.set_index('page')
print('V7.5:',V75)
print('Pages:',len(pages),'declared source groups:',manifest.source_group.nunique())


## 3. Load frozen arrays and reproduce V8.0b labels

Both context arms use identical truth labels and candidate eligibility. Clean truth is retained only for development labels and scoring.


In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def truth_maps(clean):
    ink=printed_ink(clean)
    k=np.ones((2*TRUTH_TOLERANCE+1,2*TRUTH_TOLERANCE+1),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    eroded=cv2.erode(ink.astype(np.uint8),k)>0
    boundary=np.logical_xor(near,eroded)
    return ink,near,boundary

def background_surface(observed,mask,sigma=45.0):
    visible=(~mask)&(observed>=200)
    num=cv2.GaussianBlur(observed.astype(np.float32)*visible,(0,0),sigma)
    den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

runtime={}; audit_rows=[]
required_arrays=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
    with np.load(cache[page]) as z:
        missing=[k for k in required_arrays if k not in z.files]
        if missing: raise RuntimeError(f'{page}: missing cached arrays {missing}')
        a={k:z[k] for k in required_arrays}
    clean=cv2.imread(str(clean_map[page]),0)
    observed=cv2.imread(str(observed_map[page]),0)
    if clean is None or observed is None or clean.shape!=observed.shape:
        raise RuntimeError(f'{page}: clean/observed read or shape failure')
    if any(v.shape!=observed.shape for v in a.values()): raise RuntimeError(f'{page}: cache shape mismatch')
    mask=a['mask']>0
    if not np.array_equal(a['geometry_t80'][~mask],observed[~mask]): raise RuntimeError(f'{page}: outside-mask V7.8 drift')
    ink,near,boundary=truth_maps(clean)
    eligible=mask&(observed<DARK_THRESHOLD)&~(a['g80_preserve']>0)
    label=np.full(observed.shape,-1,np.int8)
    label[eligible&near&~boundary]=1
    label[eligible&~near&~boundary]=0
    label[eligible&boundary]=2
    paper=background_surface(observed,mask)
    normalised=cv2.divide(observed,np.maximum(paper,1),scale=255)
    edge=cv2.Canny(observed,70,170)>0
    depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5)
    runtime[page]={
        'obs':observed,'norm':normalised,'mask':mask,'depth':depth,'edge':edge,'label':label,
        'baseline':a['geometry_t80'],'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,
        'ambiguous':a['g80_ambiguous']>0,'horizontal':a['g80_horizontal']>0,
        'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,
    }
    audit_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'eligible':int(eligible.sum()),
                       'print':int((label==1).sum()),'annotation':int((label==0).sum()),
                       'uncertain_truth':int((label==2).sum()),'cache_sha256':sha256_file(cache[page])})
pd.DataFrame(audit_rows).to_csv(AUDIT/'label_and_cache_audit.csv',index=False)
print(pd.DataFrame(audit_rows)[['eligible','print','annotation','uncertain_truth']].sum())


## 4. Reproduce the exact paired sample and fold map

A hash mismatch indicates that V8.0c is not using the same centre rows/folds as V8.0b and the run stops.


In [ ]:
rng=np.random.default_rng(SEED); rows=[]
for page,z in runtime.items():
    for cls in [0,1]:
        ys,xs=np.where(z['label']==cls)
        idx=np.arange(len(ys))
        take=idx if len(idx)<=MAX_PER_PAGE_CLASS else rng.choice(idx,MAX_PER_PAGE_CLASS,replace=False)
        for j in take:
            rows.append({'page':page,'source_group':str(meta.loc[page,'source_group']),
                         'y':int(ys[j]),'x':int(xs[j]),'truth':cls})
centres=pd.DataFrame(rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
centres['sample_id']=np.arange(len(centres))
groups=centres.source_group
n_groups=groups.nunique()
splitter=LeaveOneGroupOut() if n_groups<=10 else GroupKFold(n_splits=min(10,n_groups))
fold=np.full(len(centres),-1,int); split_rows=[]
for f,(tr,te) in enumerate(splitter.split(centres,centres.truth,groups),1):
    train_groups=set(groups.iloc[tr]); test_groups=set(groups.iloc[te]); overlap=train_groups&test_groups
    if overlap: raise RuntimeError(f'Fold {f}: source-group leakage {overlap}')
    fold[te]=f
    split_rows.append({'fold':f,'train_groups':len(train_groups),'test_groups':len(test_groups),
                       'overlap_groups':len(overlap),'test_rows':len(te),'test_pages':centres.iloc[te].page.nunique()})
centres['fold']=fold
fold_payload=centres[['sample_id','page','source_group','fold']].to_dict('records')
fold_hash=object_hash(fold_payload)
if fold_hash!=EXPECTED_V8B_FOLD_MAP_SHA256:
    raise RuntimeError(f'Fold-map hash mismatch: {fold_hash} != {EXPECTED_V8B_FOLD_MAP_SHA256}')
centres.to_csv(TABLES/'paired_sample_manifest.csv',index=False)
pd.DataFrame(split_rows).to_csv(AUDIT/'source_group_split_audit.csv',index=False)
(AUDIT/'fold_map.json').write_text(json.dumps({'sha256':fold_hash,'rows':fold_payload},indent=2))
print('Rows:',len(centres),'stable PRINT:',int(centres.truth.sum()),'fold hash:',fold_hash)


## 5. Create the shared observable channels

Channel values and preprocessing reproduce V8.0b. Only the extracted spatial extent differs.


In [ ]:
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
MAX_HALF=max(PATCH_CONTROL,PATCH_TREATMENT)//2

def prepare_channel(z,key):
    a=z[key].astype(np.float32)
    if key in ['obs','norm']: a=a/255.0
    return np.pad(a,MAX_HALF,mode='reflect')

padded={page:{key:prepare_channel(z,key) for key in CHANNELS} for page,z in runtime.items()}

def patch_tensor(row,patch):
    half=patch//2
    # Coordinates in arrays padded by MAX_HALF.
    cy=row.y+MAX_HALF; cx=row.x+MAX_HALF
    return np.stack([padded[row.page][k][cy-half:cy+half+1,cx-half:cx+half+1] for k in CHANNELS]).astype(np.float32)

# Exact central-pixel identity check for both contexts on a deterministic sample.
for j in np.linspace(0,len(centres)-1,100,dtype=int):
    r=centres.iloc[j]
    for patch in [PATCH_CONTROL,PATCH_TREATMENT]:
        x=patch_tensor(r,patch)
        assert x.shape==(len(CHANNELS),patch,patch)
        assert np.isclose(x[0,patch//2,patch//2],runtime[r.page]['obs'][r.y,r.x]/255.0)
print('Channel and centre checks passed.')


## 6. Fixed CNN and matched datasets

The architecture, optimiser, weighted loss and epoch schedule are identical for both contexts. `loss.detach().item()` is used only for logging and avoids retaining the autograd graph.


In [ ]:
class PatchDataset(Dataset):
    def __init__(self,indices,patch): self.indices=np.asarray(indices); self.patch=patch
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i]); row=centres.iloc[j]
        return torch.from_numpy(patch_tensor(row,self.patch)),torch.tensor(int(row.truth),dtype=torch.long)

class TopologyCNN(nn.Module):
    def __init__(self,n_channels):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(n_channels,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),
            nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),
            nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.head=nn.Linear(64,2)
    def forward(self,x): return self.head(self.net(x).flatten(1))

def reset_determinism():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

def train_model(train_indices,patch,fold):
    reset_determinism()
    model=TopologyCNN(len(CHANNELS)).to(DEVICE)
    optimiser=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    counts=np.bincount(centres.iloc[train_indices].truth,minlength=2)
    weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=weights)
    generator=torch.Generator(); generator.manual_seed(SEED)
    loader=DataLoader(PatchDataset(train_indices,patch),batch_size=BATCH,shuffle=True,
                      num_workers=2,pin_memory=True,generator=generator)
    history=[]; model.train()
    for epoch in range(EPOCHS):
        total=0.0
        for xb,yb in loader:
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss=loss_fn(model(xb),yb); loss.backward(); optimiser.step()
            total += loss.detach().item()*len(yb)
        mean_loss=total/len(train_indices)
        history.append({'fold':fold,'patch':patch,'epoch':epoch+1,'mean_loss':mean_loss})
        print(f'patch={patch} fold={fold} epoch={epoch+1}/{EPOCHS} loss={mean_loss:.6f}')
    return model,history

def predict_model(model,test_indices,patch):
    model.eval(); output=[]
    loader=DataLoader(PatchDataset(test_indices,patch),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
    with torch.no_grad():
        for xb,_ in loader:
            output.extend(torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,1].cpu().numpy())
    return np.asarray(output)


## 7. Matched source-group OOF training: 33×33 versus 65×65


In [ ]:
raw_33=np.full(len(centres),np.nan)
raw_65=np.full(len(centres),np.nan)
training_history=[]; fold_audit=[]

for f in sorted(centres.fold.unique()):
    test=np.where(centres.fold.to_numpy()==f)[0]
    train=np.where(centres.fold.to_numpy()!=f)[0]
    for patch,target,name in [(PATCH_CONTROL,raw_33,'cnn_33'),(PATCH_TREATMENT,raw_65,'cnn_65')]:
        model,history=train_model(train,patch,f)
        target[test]=predict_model(model,test,patch)
        training_history.extend(history)
        model_path=MODELS/f'{name}_fold_{f}.pt'
        torch.save(model.state_dict(),model_path)
        fold_audit.append({'fold':int(f),'arm':name,'patch':patch,'train_rows':len(train),'test_rows':len(test),
                           'model_sha256':sha256_file(model_path)})
        del model; gc.collect(); torch.cuda.empty_cache()

pd.DataFrame(training_history).to_csv(TABLES/'training_history.csv',index=False)
pd.DataFrame(fold_audit).to_csv(AUDIT/'fold_training_audit.csv',index=False)
if not np.isfinite(raw_33).all() or not np.isfinite(raw_65).all(): raise RuntimeError('Missing/non-finite OOF scores.')
print('All OOF scores complete.')


## 8. Identical cross-fitted calibration and registered decision rule


In [ ]:
def crossfit_calibrate(raw):
    calibrated=np.full(len(raw),np.nan)
    for f in sorted(centres.fold.unique()):
        test=np.where(centres.fold.to_numpy()==f)[0]
        train=np.where(centres.fold.to_numpy()!=f)[0]
        iso=IsotonicRegression(out_of_bounds='clip').fit(raw[train],centres.truth.to_numpy()[train])
        calibrated[test]=iso.predict(raw[test])
    return calibrated

def decide(prob,print_threshold=PRINT_THRESHOLD):
    prob=np.asarray(prob); p_annotation=1-prob
    out=np.full(len(prob),'UNCERTAIN',object)
    out[p_annotation>=ANNOTATION_THRESHOLD]='ANNOTATION'
    out[(prob>=print_threshold)&((prob-p_annotation)>=PRINT_MARGIN)]='PRINT'
    return out

p33=crossfit_calibrate(raw_33); p65=crossfit_calibrate(raw_65)
paired=centres.copy()
paired['raw_33']=raw_33; paired['p_33']=p33; paired['decision_33']=decide(p33)
paired['raw_65']=raw_65; paired['p_65']=p65; paired['decision_65']=decide(p65)
paired.to_parquet(TABLES/'paired_oof_predictions.parquet',index=False)
paired.to_csv(TABLES/'paired_oof_predictions.csv',index=False)
print(paired[['decision_33','decision_65']].apply(pd.Series.value_counts).fillna(0))


## 9. Ranking and calibration diagnostics

These are descriptive diagnostics. They do not authorise threshold changes.


In [ ]:
def diagnostics(raw,prob,name):
    return {
        'arm':name,'rows':len(prob),'print_prevalence':float(centres.truth.mean()),
        'raw_roc_auc':roc_auc_score(centres.truth,raw),
        'raw_average_precision':average_precision_score(centres.truth,raw),
        'calibrated_roc_auc':roc_auc_score(centres.truth,prob),
        'calibrated_average_precision':average_precision_score(centres.truth,prob),
        'brier_score':brier_score_loss(centres.truth,prob),
        'raw_max':float(np.max(raw)),'calibrated_max':float(np.max(prob)),
        'unique_calibrated_values':int(pd.Series(prob).nunique())}

diag=pd.DataFrame([diagnostics(raw_33,p33,'cnn_33'),diagnostics(raw_65,p65,'cnn_65')])
diag.to_csv(TABLES/'ranking_and_calibration_summary.csv',index=False)
per_fold=[]
for f,g in paired.groupby('fold'):
    row={'fold':int(f),'rows':len(g),'pages':g.page.nunique(),'source_groups':g.source_group.nunique(),
         'print_prevalence':float(g.truth.mean())}
    for suffix in ['33','65']:
        row[f'raw_roc_{suffix}']=roc_auc_score(g.truth,g[f'raw_{suffix}']) if g.truth.nunique()==2 else np.nan
        row[f'raw_ap_{suffix}']=average_precision_score(g.truth,g[f'raw_{suffix}']) if g.truth.sum()>0 else np.nan
        row[f'p_max_{suffix}']=float(g[f'p_{suffix}'].max())
    per_fold.append(row)
pd.DataFrame(per_fold).to_csv(TABLES/'ranking_by_fold.csv',index=False)
display(diag)


## 10. Paired safe-coverage frontier

The common threshold grid and registered rule are unchanged. A zero-selection arm is reported as non-operational evidence, not a safety pass.


In [ ]:
def frontier(prob,name):
    rows=[]; truth=centres.truth.to_numpy()
    for threshold in THRESHOLD_GRID:
        selected=(prob>=threshold)&((2*prob-1)>=PRINT_MARGIN)
        tp=int(np.sum(selected&(truth==1))); fp=int(np.sum(selected&(truth==0))); n=int(selected.sum())
        rows.append({'arm':name,'threshold':float(threshold),'selected':n,'true_print':tp,'false_annotation':fp,
                     'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(n,1),
                     'wilson95_upper':wilson_upper(fp,n),'coverage':n/len(prob),'utility':tp-5*fp,
                     'pages':centres.loc[selected,'page'].nunique(),
                     'source_groups':centres.loc[selected,'source_group'].nunique()})
    return pd.DataFrame(rows)

frontier_table=pd.concat([frontier(p33,'cnn_33'),frontier(p65,'cnn_65')],ignore_index=True)
frontier_table.to_csv(TABLES/'paired_frontier_comparison.csv',index=False)
registered=frontier_table[np.isclose(frontier_table.threshold,PRINT_THRESHOLD)].copy()
registered.to_csv(TABLES/'registered_operating_point_comparison.csv',index=False)
display(registered)


## 11. Paired disagreement queues

These queues identify whether larger context recovers PRINT candidates lost by 33×33 and whether it introduces new false PRINT decisions.


In [ ]:
print33=paired.decision_33=='PRINT'; print65=paired.decision_65=='PRINT'
paired[print65&~print33].to_csv(TABLES/'context65_only_print.csv',index=False)
paired[print33&~print65].to_csv(TABLES/'context33_only_print.csv',index=False)
paired[print65&print33&(paired.truth==0)].to_csv(TABLES/'shared_false_print.csv',index=False)
paired[(paired.truth==1)&(paired.p_65>paired.p_33)].sort_values('p_65',ascending=False).head(500).to_csv(TABLES/'context65_improved_true_rank.csv',index=False)
paired[(paired.truth==0)&(paired.p_65>paired.p_33)].sort_values('p_65',ascending=False).head(500).to_csv(TABLES/'context65_worsened_false_rank.csv',index=False)

# Page and source-group summaries at the registered rule.
for key in ['page','source_group']:
    records=[]
    for value,g in paired.groupby(key):
        r={key:value,'rows':len(g),'truth_print':int(g.truth.sum())}
        for suffix in ['33','65']:
            sel=g[f'decision_{suffix}']=='PRINT'
            r[f'selected_{suffix}']=int(sel.sum())
            r[f'true_print_{suffix}']=int((sel&(g.truth==1)).sum())
            r[f'false_annotation_{suffix}']=int((sel&(g.truth==0)).sum())
        records.append(r)
    pd.DataFrame(records).to_csv(TABLES/f'results_by_{key}.csv',index=False)
print('Disagreement and grouped summaries exported.')


## 12. V8.0c interpretation gate

V8.0c supports the larger-context hypothesis only if 65×65 improves out-of-fold ranking and provides a better safe-coverage frontier without worsening false preservation or source-group stability. It does not freeze or promote either model.

Full eligible-pixel scoring remains a later mandatory step only if this paired screening result is favourable.


In [ ]:
manifest_out={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.0c-controlled-65px-context-ablation',
    'development_only':True,
    'v7_5_reference':str(V75),
    'development_score_manifest':str(MANIFEST_PATH),
    'manifest_sha256':sha256_file(MANIFEST_PATH),
    'fold_map_sha256':fold_hash,
    'expected_v8b_fold_map_sha256':EXPECTED_V8B_FOLD_MAP_SHA256,
    'pages':len(pages),'declared_source_groups':int(manifest.source_group.nunique()),
    'scored_source_groups':int(centres.source_group.nunique()),'paired_rows':len(centres),
    'control_patch':PATCH_CONTROL,'treatment_patch':PATCH_TREATMENT,'channels':CHANNELS,
    'architecture':'same fixed compact CNN for both arms',
    'same_labels':True,'same_samples':True,'same_folds':True,'same_channels':True,
    'same_architecture':True,'same_loss':True,'same_training_schedule':True,
    'same_calibration':True,'same_decision_rule':True,
    'only_intended_change':'spatial context 33x33 to 65x65',
    'full_pixel_scoring_completed':False,'freeze_permitted':False,'independent_data_used':False,
    'software':{'python':sys.version,'torch':torch.__version__,'opencv':cv2.__version__,
                'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest_out,indent=2))
print(json.dumps(manifest_out,indent=2))


## Outputs to share for review

Please share:

- `experiment_manifest.json`
- `audit/source_group_split_audit.csv`
- `audit/fold_training_audit.csv`
- `tables/ranking_and_calibration_summary.csv`
- `tables/ranking_by_fold.csv`
- `tables/paired_frontier_comparison.csv`
- `tables/registered_operating_point_comparison.csv`
- `tables/context65_only_print.csv`
- `tables/context33_only_print.csv`
- `tables/shared_false_print.csv`
- `tables/results_by_page.csv`
- `tables/results_by_source_group.csv`

The large paired prediction file can remain in Drive unless detailed forensic analysis is needed.
